In [1]:
import scipy.io
import pandas as pd
import numpy as np

In [2]:
mat_data = scipy.io.loadmat('./rst/exp3_gen/rst_ExpTrajGenPos.mat')
genposall = mat_data['genposall']
genposall

array([[array([[2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2350.        , 1500.        , 1650.        , 1500.        ],
               [2347.70632731, 1500.        , 1650.        , 1500.        ],
               [2347.70632731, 1500.        , 1650.        , 1500.        ],
               [2351.94497182, 1531.45715208, 1649.68      , 1498.25      ],

In [3]:
print(type(genposall), genposall.shape)

<class 'numpy.ndarray'> (8, 8)


In [4]:
force_id_label = pd.read_excel("./charades_traj_summary.xlsx", sheet_name="selected_v3_forceA", header=None)
ID_LIST_FORCE, LABEL_LIST_FORCE = force_id_label[0], force_id_label[1]
traj_id_label = pd.read_excel("./charades_traj_summary.xlsx", sheet_name="selected_v3_trajB", header=None)
ID_LIST_TRAJ, LABEL_LIST_TRAJ = traj_id_label[0], traj_id_label[1]

In [5]:
force_a_id = []
force_a_label = []
traj_b_id= []
traj_b_label = []
X1_Values = []
Y1_Values = []
X2_Values = []
Y2_Values = []
rows, cols = genposall.shape # correspond to i and fj

outBoundVideo = {key: [] for key in LABEL_LIST_FORCE}

for fj in range(cols):
    for i in range(rows):
        current_cell = genposall[i, fj]
        force_a_id.append(ID_LIST_FORCE[fj])
        force_a_label.append(LABEL_LIST_FORCE[fj])
        traj_b_id.append(ID_LIST_TRAJ[i])
        traj_b_label.append(LABEL_LIST_TRAJ[i])

        # check if needs mirroring
        first_x1 = current_cell[:, 0][0]
        if first_x1 == 950:
            print(f"{ID_LIST_FORCE[fj]}_{LABEL_LIST_FORCE[fj]}_{ID_LIST_TRAJ[i]}_{LABEL_LIST_TRAJ[i]} needs to be mirrored")
            x_center = 1650
            mirror_const = 2 * x_center  # 3300
            current_cell[:, 0] = mirror_const - current_cell[:, 0]
            current_cell[:, 2] = mirror_const - current_cell[:, 2]
        
        outBound = False
        # check if outside bound
        if any(current_cell[:, 0] < 0) or any(current_cell[:, 0] > 4000):
            outBound = True
            print(f"{ID_LIST_FORCE[fj]}_{LABEL_LIST_FORCE[fj]}_{ID_LIST_TRAJ[i]}_{LABEL_LIST_TRAJ[i]} agent A out horizontal bound")
        if any(current_cell[:, 1] < 0) or any(current_cell[:, 1] > 3000):
            outBound = True
            print(f"{ID_LIST_FORCE[fj]}_{LABEL_LIST_FORCE[fj]}_{ID_LIST_TRAJ[i]}_{LABEL_LIST_TRAJ[i]} agent A out vertical bound")
        if any(current_cell[:, 2] < 0) or any(current_cell[:, 2] > 4000):
            outBound = True
            print(f"{ID_LIST_FORCE[fj]}_{LABEL_LIST_FORCE[fj]}_{ID_LIST_TRAJ[i]}_{LABEL_LIST_TRAJ[i]} agent B out horizontal bound")
        if any(current_cell[:, 3] < 0) or any(current_cell[:, 3] > 3000):
            outBound = True
            print(f"{ID_LIST_FORCE[fj]}_{LABEL_LIST_FORCE[fj]}_{ID_LIST_TRAJ[i]}_{LABEL_LIST_TRAJ[i]} agent B out vertical bound")
        if outBound:
            outBoundVideo[LABEL_LIST_FORCE[fj]].append(ID_LIST_TRAJ[i])
        
        X1_Values.append(" ".join(map(str, current_cell[:, 0]))) # each trajectory is separated by space
        Y1_Values.append(" ".join(map(str, current_cell[:, 1])))
        X2_Values.append(" ".join(map(str, current_cell[:, 2])))
        Y2_Values.append(" ".join(map(str, current_cell[:, 3])))

6017_encircle_1105_escape needs to be mirrored
5902_hit_1105_escape needs to be mirrored
5986_fight_1105_escape agent A out vertical bound
5986_fight_2029_encircle agent A out vertical bound
5986_fight_5789_capture agent A out horizontal bound
5986_fight_5789_capture agent A out vertical bound
5986_fight_5999_mimic agent A out vertical bound
5787_accompany_1105_escape needs to be mirrored
6079_bother_1051_herd needs to be mirrored
6079_bother_5098_argue with needs to be mirrored
6079_bother_6054_accompany needs to be mirrored
6035_capture_1105_escape needs to be mirrored


In [6]:
{k: len(v) for (k,v) in outBoundVideo.items()}

{'encircle': 0,
 'scratch': 0,
 'leave': 0,
 'hit': 0,
 'fight': 4,
 'accompany': 0,
 'bother': 0,
 'capture': 0}

In [7]:
union_list = []
[union_list.extend(value_list) for k,value_list in outBoundVideo.items() if k not in ["fight"]] #ignore
union_list

[]

In [8]:
len(np.unique(union_list))

0

In [9]:
trajectory_df = pd.DataFrame({"force_a_id": force_a_id,
                              "force_a_label": force_a_label,
                              "traj_b_id": traj_b_id,
                              "traj_b_label": traj_b_label,
                              "Shape1_Name Shape2_Name": ["bigCircle littleCircle" for _ in range(len(traj_b_id))],
                              "X1_Values": X1_Values,
                              "Y1_Values": Y1_Values,
                              "X2_Values": X2_Values,
                              "Y2_Values": Y2_Values})

In [10]:
# trajectory_df.to_csv("./../../stimuliPrep/generated_force_exp3.csv", index=False)